# Setup


In [1]:
import numpy as np
import pandas as pd

In [2]:
import sys
sys.path.append(r"/")
sys.path.append(r"/")

from provenance_coo import Provenance, print_prov_result, trace

prov = Provenance()

# Dataset
The [German Credit Dataset](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data) uses a set of attributes to predict whether an individual's credit risk is good or bad. 

Input:  1000 records, 20 features (13 categorical, 7 numerical)

Target: 1000 records, 2 categories (1 = Good, 2 = Bad)






In [3]:
# Load dataset
from ucimlrepo import fetch_ucirepo
statlog_german_credit_data = fetch_ucirepo(id=144)
X = statlog_german_credit_data.data.features
y = statlog_german_credit_data.data.targets

variables = statlog_german_credit_data.variables
categorical_features = variables.loc[(variables['type'] != 'Integer') & (variables['role'] == 'Feature'), 'name'].tolist()
numerical_features = variables.loc[variables['type'] == 'Integer', 'name'].tolist()

In [4]:
# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline

In [5]:
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_train_scaled = X_train.copy()
# X_train_scaled[numerical_features] = scaler.fit_transform(X_train_scaled[numerical_features])

# from sklearn.preprocessing import OneHotEncoder
# encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# X_train_categorical_encoded = encoder.fit_transform(X_train_scaled[categorical_features])  # np.ndarray
# print(X_train_categorical_encoded)
# # Convert back to DataFrame
# encoded_features = encoder.get_feature_names_out()
# X_train_categorical_encoded = pd.DataFrame(X_train_categorical_encoded, columns=encoded_features, index=X_train_scaled.index)
# X_train_encoded = pd.concat([X_train_scaled.drop(columns=categorical_features), X_train_categorical_encoded], axis=1)

# from sklearn.feature_selection import SelectKBest, chi2
# k = 40
# selector = SelectKBest(score_func=chi2, k=k)
# X_train_categorical_selected = selector.fit_transform(X_train_categorical_encoded, y_train)
# selected_features = pd.Index(encoded_features)[selector.get_support()].to_list()

# X_train_categorical_selected = selector.transform(X_train_encoded[encoded_features])
# # Convert back to DataFrame
# selected_features = pd.Index(encoded_features)[selector.get_support()].to_list()
# X_train_selected = X_train_encoded[numerical_features + selected_features]

# from imblearn.over_sampling import RandomOverSampler  
# ros = RandomOverSampler(random_state=42)
# X_train_resampled, y_train_resampled = ros.fit_resample(X_train_selected, y_train)

# from sklearn.ensemble import RandomForestClassifier
# clf = RandomForestClassifier(n_estimators=100, random_state=42)
# clf.fit(X_train_resampled, y_train_resampled.squeeze())

In [6]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import RandomForestClassifier

numerical_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
    ('selector', SelectKBest(score_func=chi2, k=40))
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('sampler', RandomOverSampler(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train.squeeze())

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Attribute2', 'Attribute5',
                                                   'Attribute8', 'Attribute11',
                                                   'Attribute13', 'Attribute16',
                                                   'Attribute18']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False)),
                                                                  ('selector',
                                                                   SelectKBest(k=40,
                                                                               score_func=<function chi2 at 0x00000233B88C8E00>))]),
                                                  ['Attribute1', 'Attribute3',
                                                   'Attribute4', 'Attribute6',
                                                   'Attribute7', 'Attribute9',
                                                   'Attribute10', 'Attribute12',
                                                   'Attribute14', 'Attribute15',
                                                   'Attribute17', 'Attribute19',
                                                   'Attribute20'])])),
                ('sampler', RandomOverSampler(random_state=42)),
                ('classifier', RandomForestClassifier(random_state=42))])

# Test

In [7]:
X_test

,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,Attribute11,Attribute12,Attribute13,Attribute14,Attribute15,Attribute16,Attribute17,Attribute18,Attribute19,Attribute20
521,A11,18,A32,A43,3190,A61,A73,2,A92,A101,2,A121,24,A143,A152,1,A173,1,A191,A201
737,A11,18,A32,A40,4380,A62,A73,3,A93,A101,4,A123,35,A143,A152,1,A172,2,A192,A201
740,A11,24,A31,A40,2325,A62,A74,2,A93,A101,3,A123,32,A141,A152,1,A173,1,A191,A201
660,A13,12,A32,A43,1297,A61,A73,3,A94,A101,4,A121,23,A143,A151,1,A173,1,A191,A201
411,A14,33,A34,A41,7253,A61,A74,3,A93,A101,2,A123,35,A143,A152,2,A174,1,A192,A201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,A14,24,A32,A43,3235,A63,A75,3,A91,A101,2,A123,26,A143,A152,1,A174,1,A192,A201
332,A12,60,A32,A40,7408,A62,A72,4,A92,A101,2,A122,24,A143,A152,1,A174,1,A191,A201
208,A11,24,A32,A49,6568,A61,A73,2,A94,A101,2,A123,21,A142,A152,1,A172,1,A191,A201
613,A11,24,A31,A41,3632,A61,A73,1,A92,A103,4,A123,22,A141,A151,1,A173,1,A191,A202


In [8]:
y_test

,class
521,2
737,1
740,1
660,1
411,1
...,...
408,1
332,2
208,1
613,1


# 1. Data Transformation

Standardization for numerical features.

In [9]:
scaler = model.named_steps['preprocessor'].named_transformers_['num'].named_steps['scaler']
X_test_numerical_scaled = scaler.transform(X_test[numerical_features])
# Convert back to DataFrame
X_test_scaled = pd.DataFrame(
    np.hstack([X_test_numerical_scaled, X_test[categorical_features]]),
    columns=numerical_features + categorical_features,
    index=X_test.index
)
X_test_scaled

,Attribute2,Attribute5,Attribute8,Attribute11,Attribute13,Attribute16,Attribute18,Attribute1,Attribute3,Attribute4,Attribute6,Attribute7,Attribute9,Attribute10,Attribute12,Attribute14,Attribute15,Attribute17,Attribute19,Attribute20
521,-0.262292,-0.058908,-0.860109,-0.766124,-1.01353,-0.710931,-0.409736,A11,A32,A43,A61,A73,A92,A101,A121,A143,A152,A173,A191,A201
737,-0.262292,0.351952,0.031196,1.044509,-0.048994,-0.710931,2.440599,A11,A32,A40,A62,A73,A93,A101,A123,A143,A152,A172,A192,A201
740,0.24619,-0.357558,-0.860109,0.139192,-0.312049,-0.710931,-0.409736,A11,A31,A40,A62,A74,A93,A101,A123,A141,A152,A173,A191,A201
660,-0.770774,-0.712486,0.031196,1.044509,-1.101215,-0.710931,-0.409736,A13,A32,A43,A61,A73,A94,A101,A121,A143,A151,A173,A191,A201
411,1.008913,1.343886,0.031196,-0.766124,-0.048994,1.017777,-0.409736,A14,A34,A41,A61,A74,A93,A101,A123,A143,A152,A174,A192,A201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,0.24619,-0.043371,0.031196,-0.766124,-0.838159,-0.710931,-0.409736,A14,A32,A43,A63,A75,A91,A101,A123,A143,A152,A174,A192,A201
332,3.297082,1.397401,0.9225,-0.766124,-1.01353,-0.710931,-0.409736,A12,A32,A40,A62,A72,A92,A101,A122,A143,A152,A174,A191,A201
208,0.24619,1.107382,-0.860109,-0.766124,-1.276585,-0.710931,-0.409736,A11,A32,A49,A61,A73,A94,A101,A123,A142,A152,A172,A191,A201
613,0.24619,0.093697,-1.751413,1.044509,-1.1889,-0.710931,-0.409736,A11,A31,A41,A61,A73,A92,A103,A123,A141,A151,A173,A191,A202


In [11]:
result1, runtime = prov.capture(X_test, X_test_scaled)
print_prov_result(X_test, X_test_scaled, result1)

Runtime: 0.0040 seconds

-- Record tensor --

<COOrdinate sparse matrix of dtype 'int8'
	with 200 stored elements and shape (200, 200)>
  Coords	Values
  (0, 0)	1
  (1, 1)	1
  (2, 2)	1
  (3, 3)	1
  (4, 4)	1
  (5, 5)	1
  (6, 6)	1
  (7, 7)	1
  (8, 8)	1
  (9, 9)	1
  (10, 10)	1
  (11, 11)	1
  (12, 12)	1
  (13, 13)	1
  (14, 14)	1
  (15, 15)	1
  (16, 16)	1
  (17, 17)	1
  (18, 18)	1
  (19, 19)	1
  (20, 20)	1
  (21, 21)	1
  (22, 22)	1
  (23, 23)	1
  (24, 24)	1
  :	:
  (175, 175)	1
  (176, 176)	1
  (177, 177)	1
  (178, 178)	1
  (179, 179)	1
  (180, 180)	1
  (181, 181)	1
  (182, 182)	1
  (183, 183)	1
  (184, 184)	1
  (185, 185)	1
  (186, 186)	1
  (187, 187)	1
  (188, 188)	1
  (189, 189)	1
  (190, 190)	1
  (191, 191)	1
  (192, 192)	1
  (193, 193)	1
  (194, 194)	1
  (195, 195)	1
  (196, 196)	1
  (197, 197)	1
  (198, 198)	1
  (199, 199)	1

-- Record examples --

D_out[0] is from D_in[0]: 
	    Attribute2 Attribute5 Attribute8 Attribute11 Attribute13 Attribute16  \
521  -0.262292  -0.058908  -0.8601

# 2. Vertical Augmentation

One-Hot Encoding for categorical features.








In [12]:
# Inject an error
i_alice = 3
X_test_scaled.iloc[i_alice, 7] = "A11"
X_test_scaled

,Attribute2,Attribute5,Attribute8,Attribute11,Attribute13,Attribute16,Attribute18,Attribute1,Attribute3,Attribute4,Attribute6,Attribute7,Attribute9,Attribute10,Attribute12,Attribute14,Attribute15,Attribute17,Attribute19,Attribute20
521,-0.262292,-0.058908,-0.860109,-0.766124,-1.01353,-0.710931,-0.409736,A11,A32,A43,A61,A73,A92,A101,A121,A143,A152,A173,A191,A201
737,-0.262292,0.351952,0.031196,1.044509,-0.048994,-0.710931,2.440599,A11,A32,A40,A62,A73,A93,A101,A123,A143,A152,A172,A192,A201
740,0.24619,-0.357558,-0.860109,0.139192,-0.312049,-0.710931,-0.409736,A11,A31,A40,A62,A74,A93,A101,A123,A141,A152,A173,A191,A201
660,-0.770774,-0.712486,0.031196,1.044509,-1.101215,-0.710931,-0.409736,A11,A32,A43,A61,A73,A94,A101,A121,A143,A151,A173,A191,A201
411,1.008913,1.343886,0.031196,-0.766124,-0.048994,1.017777,-0.409736,A14,A34,A41,A61,A74,A93,A101,A123,A143,A152,A174,A192,A201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,0.24619,-0.043371,0.031196,-0.766124,-0.838159,-0.710931,-0.409736,A14,A32,A43,A63,A75,A91,A101,A123,A143,A152,A174,A192,A201
332,3.297082,1.397401,0.9225,-0.766124,-1.01353,-0.710931,-0.409736,A12,A32,A40,A62,A72,A92,A101,A122,A143,A152,A174,A191,A201
208,0.24619,1.107382,-0.860109,-0.766124,-1.276585,-0.710931,-0.409736,A11,A32,A49,A61,A73,A94,A101,A123,A142,A152,A172,A191,A201
613,0.24619,0.093697,-1.751413,1.044509,-1.1889,-0.710931,-0.409736,A11,A31,A41,A61,A73,A92,A103,A123,A141,A151,A173,A191,A202


In [13]:
encoder = model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
X_test_categorical_encoded = encoder.transform(X_test_scaled[categorical_features])
# Convert back to DataFrame
encoded_features = encoder.get_feature_names_out()
X_test_encoded = pd.DataFrame(
    np.hstack([X_test_numerical_scaled, X_test_categorical_encoded]),
    columns=numerical_features + list(encoded_features),
    index=X_test.index
)
X_test_encoded

,Attribute2,Attribute5,Attribute8,Attribute11,Attribute13,Attribute16,Attribute18,Attribute1_A11,Attribute1_A12,Attribute1_A13,...,Attribute15_A152,Attribute15_A153,Attribute17_A171,Attribute17_A172,Attribute17_A173,Attribute17_A174,Attribute19_A191,Attribute19_A192,Attribute20_A201,Attribute20_A202
521,-0.262292,-0.058908,-0.860109,-0.766124,-1.013530,-0.710931,-0.409736,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
737,-0.262292,0.351952,0.031196,1.044509,-0.048994,-0.710931,2.440599,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
740,0.246190,-0.357558,-0.860109,0.139192,-0.312049,-0.710931,-0.409736,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
660,-0.770774,-0.712486,0.031196,1.044509,-1.101215,-0.710931,-0.409736,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
411,1.008913,1.343886,0.031196,-0.766124,-0.048994,1.017777,-0.409736,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,0.246190,-0.043371,0.031196,-0.766124,-0.838159,-0.710931,-0.409736,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
332,3.297082,1.397401,0.922500,-0.766124,-1.013530,-0.710931,-0.409736,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
208,0.246190,1.107382,-0.860109,-0.766124,-1.276585,-0.710931,-0.409736,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
613,0.246190,0.093697,-1.751413,1.044509,-1.188900,-0.710931,-0.409736,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


# Provenance



In [14]:
def get_onehot_mapping(encoder):
    mapping = {}
    for col_in, cats in zip(encoder.feature_names_in_, encoder.categories_):
        mapping[col_in] = [f"{col_in}_{cat}" for cat in cats]
    return mapping

mapping = get_onehot_mapping(encoder)
result2, runtime = prov.capture(X_test_scaled, X_test_encoded, column_mapping=mapping)
print_prov_result(X_test_scaled, X_test_encoded, result2)

Runtime: 0.0009 seconds

-- Record tensor --

<COOrdinate sparse matrix of dtype 'int8'
	with 200 stored elements and shape (200, 200)>
  Coords	Values
  (0, 0)	1
  (1, 1)	1
  (2, 2)	1
  (3, 3)	1
  (4, 4)	1
  (5, 5)	1
  (6, 6)	1
  (7, 7)	1
  (8, 8)	1
  (9, 9)	1
  (10, 10)	1
  (11, 11)	1
  (12, 12)	1
  (13, 13)	1
  (14, 14)	1
  (15, 15)	1
  (16, 16)	1
  (17, 17)	1
  (18, 18)	1
  (19, 19)	1
  (20, 20)	1
  (21, 21)	1
  (22, 22)	1
  (23, 23)	1
  (24, 24)	1
  :	:
  (175, 175)	1
  (176, 176)	1
  (177, 177)	1
  (178, 178)	1
  (179, 179)	1
  (180, 180)	1
  (181, 181)	1
  (182, 182)	1
  (183, 183)	1
  (184, 184)	1
  (185, 185)	1
  (186, 186)	1
  (187, 187)	1
  (188, 188)	1
  (189, 189)	1
  (190, 190)	1
  (191, 191)	1
  (192, 192)	1
  (193, 193)	1
  (194, 194)	1
  (195, 195)	1
  (196, 196)	1
  (197, 197)	1
  (198, 198)	1
  (199, 199)	1

-- Record examples --

D_out[0] is from D_in[0]: 
	     Attribute2  Attribute5  Attribute8  Attribute11  Attribute13  \
521   -0.262292   -0.058908   -0.860109  

# 3.Vertical Reduction

Select features by Chi-Square.  
Features with p-values <= 0.05 are considered statistically significant, indicating that they are likely associated with the target variable rather than occurring by chance.

In [15]:
selector = model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['selector']
X_test_categorical_selected = selector.transform(X_test_encoded[encoded_features].to_numpy())
# Convert back to DataFrame
selected_features = pd.Index(encoded_features)[selector.get_support()].to_list()
X_test_selected = pd.DataFrame(
    np.hstack([X_test_numerical_scaled, X_test_categorical_selected]),
    columns=numerical_features + selected_features,
    index=X_test.index
)
X_test_selected

,Attribute2,Attribute5,Attribute8,Attribute11,Attribute13,Attribute16,Attribute18,Attribute1_A11,Attribute1_A12,Attribute1_A13,...,Attribute12_A124,Attribute14_A141,Attribute14_A142,Attribute14_A143,Attribute15_A151,Attribute15_A152,Attribute15_A153,Attribute17_A172,Attribute17_A174,Attribute20_A202
521,-0.262292,-0.058908,-0.860109,-0.766124,-1.013530,-0.710931,-0.409736,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
737,-0.262292,0.351952,0.031196,1.044509,-0.048994,-0.710931,2.440599,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
740,0.246190,-0.357558,-0.860109,0.139192,-0.312049,-0.710931,-0.409736,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
660,-0.770774,-0.712486,0.031196,1.044509,-1.101215,-0.710931,-0.409736,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
411,1.008913,1.343886,0.031196,-0.766124,-0.048994,1.017777,-0.409736,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,0.246190,-0.043371,0.031196,-0.766124,-0.838159,-0.710931,-0.409736,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
332,3.297082,1.397401,0.922500,-0.766124,-1.013530,-0.710931,-0.409736,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
208,0.246190,1.107382,-0.860109,-0.766124,-1.276585,-0.710931,-0.409736,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
613,0.246190,0.093697,-1.751413,1.044509,-1.188900,-0.710931,-0.409736,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [16]:
result3, runtime = prov.capture(X_test_encoded, X_test_selected)
print_prov_result(X_test_encoded, X_test_selected, result3)

Runtime: 0.0004 seconds

-- Record tensor --

<COOrdinate sparse matrix of dtype 'int8'
	with 200 stored elements and shape (200, 200)>
  Coords	Values
  (0, 0)	1
  (1, 1)	1
  (2, 2)	1
  (3, 3)	1
  (4, 4)	1
  (5, 5)	1
  (6, 6)	1
  (7, 7)	1
  (8, 8)	1
  (9, 9)	1
  (10, 10)	1
  (11, 11)	1
  (12, 12)	1
  (13, 13)	1
  (14, 14)	1
  (15, 15)	1
  (16, 16)	1
  (17, 17)	1
  (18, 18)	1
  (19, 19)	1
  (20, 20)	1
  (21, 21)	1
  (22, 22)	1
  (23, 23)	1
  (24, 24)	1
  :	:
  (175, 175)	1
  (176, 176)	1
  (177, 177)	1
  (178, 178)	1
  (179, 179)	1
  (180, 180)	1
  (181, 181)	1
  (182, 182)	1
  (183, 183)	1
  (184, 184)	1
  (185, 185)	1
  (186, 186)	1
  (187, 187)	1
  (188, 188)	1
  (189, 189)	1
  (190, 190)	1
  (191, 191)	1
  (192, 192)	1
  (193, 193)	1
  (194, 194)	1
  (195, 195)	1
  (196, 196)	1
  (197, 197)	1
  (198, 198)	1
  (199, 199)	1

-- Record examples --

D_out[0] is from D_in[0]: 
	     Attribute2  Attribute5  Attribute8  Attribute11  Attribute13  \
521   -0.262292   -0.058908   -0.860109  

# Evaluation

In [17]:
from sklearn.metrics import classification_report

classifier = model.named_steps['classifier']
y_test_pred = classifier.predict(X_test_selected)
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           1       0.86      0.92      0.89       141
           2       0.77      0.63      0.69        59

    accuracy                           0.83       200
   macro avg       0.81      0.77      0.79       200
weighted avg       0.83      0.83      0.83       200



C:\Users\30315\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [18]:
y_test.iloc[i_alice]["class"]

np.int64(1)

In [19]:
y_test_pred[3]

np.int64(2)

# Debugging

Alice has a good credit history but was denied a loan.   
It is due to an encoding error of Attribute1 (Status of existing checking account),   
A13 (>= 200 DM / salary assignments for at least 1 year)
was mistakenly encoded as 
A11 (< 0 DM).

In [20]:
dfs = [X_test, X_test_scaled, X_test_encoded, X_test_selected]
results = [result1, result2, result3]
tensors_record = [result[0] for result in results]
tensors_attr = [result[1] for result in results]

trace_record = trace(tensors_record, direction="backward", indices=[i_alice])
# trace_record = trace(tensors_record, direction="backward", indices=[i_alice], keep_path=True)
tensors_attr = trace(tensors_attr, direction="backward")
print_prov_result(X_test, X_test_selected, (trace_record, tensors_attr))

# print(trace_record)
# print(tensors_attr)


-- Record tensor --

[[3 3]]

-- Record examples --

D_out[3] is from D_in[3]: 
	     Attribute2  Attribute5  Attribute8  Attribute11  Attribute13  \
660   -0.770774   -0.712486    0.031196     1.044509    -1.101215   

     Attribute16  Attribute18  Attribute1_A11  Attribute1_A12  Attribute1_A13  \
660    -0.710931    -0.409736             1.0             0.0             0.0   

     ...  Attribute12_A124  Attribute14_A141  Attribute14_A142  \
660  ...               0.0               0.0               0.0   

     Attribute14_A143  Attribute15_A151  Attribute15_A152  Attribute15_A153  \
660               1.0               1.0               0.0               0.0   

     Attribute17_A172  Attribute17_A174  Attribute20_A202  
660               0.0               0.0               0.0  

[1 rows x 47 columns]
	is from
	    Attribute1  Attribute2 Attribute3 Attribute4  Attribute5 Attribute6  \
660        A13          12        A32        A43        1297        A61   

    Attribute7  Attr